## Snakemake for Converting GPEP Forcing to Snakemake ##

Using [snakemake](https://snakemake.readthedocs.io/en/stable/) to wrap a meteorological data processing workflow

### Main snakemake file - gpep_to_summa

The Snakemake file below can be used to run the entire data processing workflow.
The Snakemake file is first written, then run from the command line.

#### Write the snakemake code below to file

Note that this is python code, which some additional syntax for snakemake.

In [1]:
config_file = '../config/gpep_to_summa_chena.yaml'

In [2]:
%%writefile ../gpep_to_summa.smk
''' 
gmet to summa snakemake master snakemake file

This snakemake file runs all the steps required to convert GPEP forcings to SUMMA forcings.

Original process code: Andy Wood
Adapted to Snakemake: Dave Casson

'''
from pathlib import Path
from scripts import gpep_to_summa_utils as utils

# Resolve paths from the configuration file
config = utils.resolve_paths(config, log_config = True)

# Read in all local snakemake files and rules
include: './rules/gpep_file_prep.smk'
include: './rules/remap_gpep_to_shp.smk'
include: './rules/metsim_file_prep.smk'
include: './rules/run_metsim.smk'
include: './rules/metsim_to_summa.smk'

ruleorder: 
    gpep_to_summa >
    gpep_file_prep >
    remap_gpep_to_shp >
    metsim_file_prep >
    run_metsim >
    metsim_to_summa
    
# Read all forcing files and create a list based on the output directory (i.e. ens/filename.nc)
_, file_path_list = utils.build_ensemble_list(config['gpep_forcing_dir'])

# Run the snakemake file, so that that it produces a summa input file for each of the gpep forcing files
rule gpep_to_summa:
    input:
        expand(Path(config['summa_forcing_dir'],'{forcing_file}.nc'), forcing_file = file_path_list)

Overwriting ../gpep_to_summa.smk


### Perform a Dry Run, and unlock the working directory

In [3]:
## Perform a dry run to check the workflow is configured correctly.
! snakemake --unlock -s ../gpep_to_summa.smk --configfile {config_file}
! snakemake -s ../gpep_to_summa.smk --configfile {config_file} --dry-run

Settings logged from {'base_settings': {'case_name': 'chena_test'}, 'key_directories': {'gpep_forcing_dir': '/Users/dcasson/Data/gpep/chena/ensembles/', 'working_dir': '/Users/dcasson/Data/gpep/chena/forcing/', 'summa_dir': '/Users/dcasson/Data/gpep/chena/forcing/summa/', 'gpep_to_summa_output_dir': '/Users/dcasson/Data/gpep/chena/forcing/', 'summa_forcing_dir': '/Users/dcasson/Data/gpep/chena/forcing/summa/'}, 'input_files': {'catchment_shp': '/Users/dcasson/Data/gpep/chena/gpkg/Chena_Near_Two_Rivers.gpkg', 'attribute_nc': '/Users/dcasson/Data/yukon_esp/summa/settings/attributes.nc', 'metsim_base_config': '../config/metsim_base_config.yaml'}, 'easymore': {'catchment_shp_hru_id_field': 'hruNo', 'catchment_shp_lat_id_field': 'cent_lat', 'catchment_shp_lon_id_field': 'cent_lon', 'forcing_shp': 'forcing_shp.shp', 'easymore_input_var': ['prcp', 't_max', 't_min']}, 'metsim': {'metsim_dir': '/Users/dcasson/Data/gpep/chena/forcing/metsim/', 'metsim_input_dir': '/Users/dcasson/Data/gpep/chena/

## Run the complete gmet to summa snakemake workflow

In [4]:
! snakemake -s ../gpep_to_summa.smk -c 8 --configfile {config_file}

Settings logged from {'base_settings': {'case_name': 'chena_test'}, 'key_directories': {'gpep_forcing_dir': '/Users/dcasson/Data/gpep/chena/ensembles/', 'working_dir': '/Users/dcasson/Data/gpep/chena/forcing/', 'summa_dir': '/Users/dcasson/Data/gpep/chena/forcing/summa/', 'gpep_to_summa_output_dir': '/Users/dcasson/Data/gpep/chena/forcing/', 'summa_forcing_dir': '/Users/dcasson/Data/gpep/chena/forcing/summa/'}, 'input_files': {'catchment_shp': '/Users/dcasson/Data/gpep/chena/gpkg/Chena_Near_Two_Rivers.gpkg', 'attribute_nc': '/Users/dcasson/Data/yukon_esp/summa/settings/attributes.nc', 'metsim_base_config': '../config/metsim_base_config.yaml'}, 'easymore': {'catchment_shp_hru_id_field': 'hruNo', 'catchment_shp_lat_id_field': 'cent_lat', 'catchment_shp_lon_id_field': 'cent_lon', 'forcing_shp': 'forcing_shp.shp', 'easymore_input_var': ['prcp', 't_max', 't_min']}, 'metsim': {'metsim_dir': '/Users/dcasson/Data/gpep/chena/forcing/metsim/', 'metsim_input_dir': '/Users/dcasson/Data/gpep/chena/

### Visualize the workflow rules

Note that the graphviz must be installed

In [ ]:
from IPython import display
# Buold build the rule graph
! snakemake -s ../gpep_to_summa.smk --configfile {config_file} --rulegraph | dot -Tpng > ../reports/gpep_to_summa_rule.png
# Python command to visualise the built image in our notebook
display.Image('../reports/gpep_to_summa_rule.png')


### Visualize the workflow files

Note that the graphviz must be installed

In [ ]:
# Build the file graph
! snakemake -s ../gpep_to_summa.smk --configfile {config_file} --filegraph | dot -Tpng > ../reports/gpep_to_summa_file.png
display.Image('../reports/gpep_to_summa_file.png')